In [1]:
#IMPORT LIBRARIES

import os
import sys
import re
from collections import Counter
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

print(" Libraries imported!")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

 Libraries imported!
PyTorch: 2.5.1
CUDA Available: False


In [2]:
# CONFIGURATION

# Dataset paths
BASE_DIR = "KIDO/"
IMAGES_DIR = os.path.join(BASE_DIR, "Images", "Emotion")
TEXTS_DIR = os.path.join(BASE_DIR, "Texts", "Emotion")

TRAIN_CSV = os.path.join(TEXTS_DIR, "Emotion_Train.csv")
TEST_CSV = os.path.join(TEXTS_DIR, "Emotion_Test.csv")

# Model save directory
MODEL_DIR = "Emotion_Models/"
os.makedirs(MODEL_DIR, exist_ok=True)

# Training parameters
BATCH_SIZE = 16
EPOCHS = 15
EARLY_STOPPING_PATIENCE = 3
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
MAX_TEXT_LENGTH = 50
EMBEDDING_DIM = 300

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


print(" CONTINUATION TRAINING - SQUEEZENET MODELS")
print(f"Model Save Dir:  {MODEL_DIR}")
print(f"Device:          {DEVICE}")
print(f"Epochs:          {EPOCHS}")
print(f"Patience:        {EARLY_STOPPING_PATIENCE}")

 CONTINUATION TRAINING - SQUEEZENET MODELS
Model Save Dir:  Emotion_Models/
Device:          cpu
Epochs:          15
Patience:        3


In [3]:
#PRE-DOWNLOAD SQUEEZENET WEIGHTS 

import torchvision.models as models
import socket
import time

# Set longer timeout
socket.setdefaulttimeout(60)

print(" DOWNLOADING SQUEEZENET WEIGHTS")

try:
    print("   Downloading squeezenet1_1...", end=" ")
    model = models.squeezenet1_1(pretrained=True)
    print(" Done!")
    print(f"   Model loaded successfully!")
except Exception as e:
    print(f" Failed: {e}")
    print("    Will train with random weights (pretrained=False)")

 DOWNLOADING SQUEEZENET WEIGHTS
   Model loaded successfully!


In [4]:
# LOAD DATA

def preprocess_text(text):
    """Clean and preprocess text."""
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def load_kido_data():
    """Load KIDO dataset from local structure."""
    data = []
    
    if not os.path.exists(TRAIN_CSV) or not os.path.exists(TEST_CSV):
        print("❌ CSV files not found!")
        return None, None, None
    
    print(" Loading CSV files...")
    train_df = pd.read_csv(TRAIN_CSV, header=None)
    test_df = pd.read_csv(TEST_CSV, header=None)
    
    print(f"   Train CSV: {len(train_df)} rows")
    print(f"   Test CSV: {len(test_df)} rows")
    
    ID_COL, TEXT_COL, LABEL_COL = 0, 2, 3
    
    print(" Processing data...")
    for split, df in [('train', train_df), ('test', test_df)]:
        success = 0
        for _, row in df.iterrows():
            try:
                img_id = str(row[ID_COL]).strip()
                text = str(row[TEXT_COL]) if pd.notna(row[TEXT_COL]) else ""
                label = str(row[LABEL_COL]).strip() if pd.notna(row[LABEL_COL]) else ""
                
                if label.lower() in ['happiness', 'happy']:
                    label = 'Happiness'
                elif label.lower() in ['sadness', 'sad']:
                    label = 'Sadness'
                else:
                    continue
                
                img_path = os.path.join(IMAGES_DIR, split, label, f"{img_id}.png")
                if not os.path.exists(img_path):
                    img_path = os.path.join(IMAGES_DIR, split, label, f"{img_id}.jpg")
                    if not os.path.exists(img_path):
                        continue
                
                data.append({
                    'image_id': img_id,
                    'image_path': os.path.join(split, label, f"{img_id}.png"),
                    'text': text,
                    'label': label.lower(),
                    'split': split
                })
                success += 1
            except:
                continue
        print(f"   {split}: {success} samples")
    
    if len(data) == 0:
        print(" No data found! Creating dummy data...")
        return create_dummy_data()
    
    df = pd.DataFrame(data)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    
    train_df, val_df = train_test_split(
        train_df, 
        test_size=0.15, 
        random_state=42, 
        stratify=train_df['label']
    )
    
    print(f"\n Final split:")
    print(f"   Training: {len(train_df)}")
    print(f"   Validation: {len(val_df)}")
    print(f"   Test: {len(test_df)}")
    
    return train_df, val_df, test_df

def create_dummy_data():
    """Create dummy data for testing."""
    import random
    data = []
    labels = ['happiness', 'sadness']
    for split, n in [('train', 500), ('val', 100), ('test', 100)]:
        for i in range(n):
            label = random.choice(labels)
            data.append({
                'image_id': f'dummy_{i}',
                'image_path': f'{split}/{label}/dummy_{i}.png',
                'text': f"Dummy text for {label}",
                'label': label,
                'split': split
            })
    df = pd.DataFrame(data)
    train_df = df[df['split'] == 'train'].reset_index(drop=True)
    val_df = df[df['split'] == 'val'].reset_index(drop=True)
    test_df = df[df['split'] == 'test'].reset_index(drop=True)
    print(f" Using DUMMY data")
    return train_df, val_df, test_df

print("\n LOADING DATA...")
train_df, val_df, test_df = load_kido_data()

if train_df is None:
    print(" Failed to load data!")
    exit()


 LOADING DATA...
 Loading CSV files...
   Train CSV: 9228 rows
   Test CSV: 1632 rows
 Processing data...
   train: 9228 samples
   test: 1632 samples

 Final split:
   Training: 7843
   Validation: 1385
   Test: 1632


In [5]:
# BUILD VOCABULARY & CREATE DATALOADERS

def build_vocabulary(texts, max_vocab=10000):
    """Build vocabulary from texts."""
    word_counts = Counter()
    for text in texts:
        word_counts.update(preprocess_text(text).split())
    
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, _ in word_counts.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    
    return vocab

def text_to_sequence(text, vocab, max_len):
    """Convert text to token sequence."""
    tokens = preprocess_text(text).split()[:max_len]
    seq = [vocab.get(token, vocab['<UNK>']) for token in tokens]
    seq += [0] * (max_len - len(seq))
    return torch.tensor(seq, dtype=torch.long)

def make_collate_fn(vocab, max_len):
    """Create collate function for DataLoader."""
    def collate_fn(batch):
        images = torch.stack([item['image'] for item in batch])
        labels = torch.stack([item['label'] for item in batch])
        texts = torch.stack([
            text_to_sequence(item['text'], vocab, max_len) for item in batch
        ])
        return {'image': images, 'text': texts, 'label': labels}
    return collate_fn

class KIDODataset(Dataset):
    """KIDO Dataset for emotion classification."""
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(IMAGES_DIR, row['image_path'])
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), color='white')
        
        if self.transform:
            image = self.transform(image)
        
        label = 0 if row['label'] == 'happiness' else 1
        text = row['text'] if pd.notna(row['text']) else ""
        
        return {
            'image': image,
            'text': text,
            'label': torch.tensor(label, dtype=torch.long)
        }

def create_dataloaders(train_df, val_df, test_df, batch_size=16):
    """Create DataLoaders."""
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])
    
    train_dataset = KIDODataset(train_df, train_transform)
    val_dataset = KIDODataset(val_df, val_transform)
    test_dataset = KIDODataset(test_df, val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f" DataLoaders created (batch_size={batch_size})")
    return train_loader, val_loader, test_loader

# Build vocabulary
print("\n BUILDING VOCABULARY...")
all_texts = train_df['text'].tolist() + val_df['text'].tolist()
vocab = build_vocabulary(all_texts)
vocab_size = len(vocab)
print(f" Vocabulary size: {vocab_size}")

# Create dataloaders
print("\n CREATING DATALOADERS...")
train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df, BATCH_SIZE
)

collate_fn = make_collate_fn(vocab, MAX_TEXT_LENGTH)

# Create custom dataloaders with collate
train_loader_custom = DataLoader(
    train_loader.dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)
val_loader_custom = DataLoader(
    val_loader.dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


 BUILDING VOCABULARY...
 Vocabulary size: 3423

 CREATING DATALOADERS...
 DataLoaders created (batch_size=16)


In [6]:
# MODEL COMPONENTS

class BiLSTMTextEncoder(nn.Module):
    """Bi-LSTM Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, 2, bidirectional=True, 
                           batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context

class GRUTextEncoder(nn.Module):
    """GRU Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden, 2, bidirectional=True, 
                         batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        gru_out, _ = self.gru(embedded)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        context = torch.sum(attn_weights * gru_out, dim=1)
        return context

def get_vision_encoder(name, pretrained=True):
    """Get vision backbone with feature dimension."""
    backbones = {
        'mobilenet_v2': (models.mobilenet_v2, 1280),
        'efficientnet_b0': (models.efficientnet_b0, 1280),
        'shufflenet_v2_x1_0': (models.shufflenet_v2_x1_0, 1024),
        'squeezenet1_1': (models.squeezenet1_1, 512),
    }
    
    if name not in backbones:
        available = list(backbones.keys())
        raise ValueError(f"Unknown model: {name}. Available: {available}")
    
    model_fn, dim = backbones[name]
    model = model_fn(pretrained=pretrained)
    
    # Remove classification head
    if hasattr(model, 'classifier'):
        model.classifier = nn.Identity()
    elif hasattr(model, 'fc'):
        model.fc = nn.Identity()
    
    # Freeze early layers
    for param in list(model.parameters())[:10]:
        param.requires_grad = False
    
    return model, dim

class MultimodalModel(nn.Module):
    """Multimodal emotion classifier."""
    def __init__(self, vision_name, text_encoder, dropout=0.5):
        super().__init__()
        
        self.vision, vdim = get_vision_encoder(vision_name)
        self.vision_proj = nn.Sequential(
            nn.Linear(vdim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        self.text_encoder = text_encoder
        self.text_proj = nn.Sequential(
            nn.Linear(text_encoder.output_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        fusion_dim = 512 + 256
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )
        
    def forward(self, images, texts):
        v = self.vision(images)
        if v.dim() > 2:
            v = v.mean([2, 3])
        v = self.vision_proj(v)
        
        t = self.text_encoder(texts)
        t = self.text_proj(t)
        
        fused = torch.cat([v, t], dim=1)
        return self.classifier(fused)
    
    def get_parameters(self):
        return self.parameters()

def create_text_encoder(config, vocab_size):
    """Create text encoder based on config."""
    if config['text_encoder'] == 'bilstm':
        return BiLSTMTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    elif config['text_encoder'] == 'gru':
        return GRUTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    else:
        raise ValueError(f"Unknown text encoder: {config['text_encoder']}")

print(" Model components ready")

 Model components ready


In [7]:
# TRAINING FUNCTIONS


def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch in tqdm(loader, desc='Training', leave=False):
        images = batch['image'].to(device)
        texts = batch['text'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(images, texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    """Validate the model."""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc='Validation', leave=False):
            images = batch['image'].to(device)
            texts = batch['text'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(images, texts)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
    
    return total_loss / len(loader), correct / total

def train_model(model, train_loader, val_loader, model_name, 
                epochs=15, patience=3):
    """Train model with early stopping."""
    
    device = DEVICE
    model.to(device)
    
    optimizer = optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    criterion = nn.CrossEntropyLoss()
    
    best_acc = 0
    best_epoch = 0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    
    print(f" TRAINING: {model_name}")
    print(f"   Device: {device}")
    print(f"   Epochs: {epochs}")
    print(f"   Patience: {patience}")

    
    for epoch in range(epochs):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step(val_acc)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_epoch = epoch + 1
            patience_counter = 0
            print(f" New best: {best_acc:.4f}")
        else:
            patience_counter += 1
            print(f"  No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f" Early stopping triggered!")
                break
    
    print(f"\n {model_name} COMPLETE!")
    print(f"   Best Accuracy: {best_acc:.4f} at epoch {best_epoch}")
    print(f"   Total epochs: {epoch+1}")

    
    return model, history, best_acc

print(" Training functions ready")

 Training functions ready


In [8]:
# SAVE FUNCTIONS

def save_model(model, optimizer, history, model_name, best_acc):
    """Save trained model to disk."""
    save_path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")
    
    checkpoint = {
        'model_name': model_name,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
        'best_accuracy': best_acc,
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'device': str(DEVICE)
    }
    
    torch.save(checkpoint, save_path)
    
    print(f" Model saved: {save_path}")
    print(f"   Best Accuracy: {best_acc:.4f}")
    print(f"   File Size: {os.path.getsize(save_path) / 1024 / 1024:.2f} MB")
    
    return save_path

def model_exists(model_name):
    """Check if model is already saved."""
    path = os.path.join(MODEL_DIR, f"{model_name}_final.pt")
    if os.path.exists(path):
        size = os.path.getsize(path) / 1024 / 1024
        print(f"  {model_name} already exists ({size:.1f} MB)")
        return True
    return False

print(" Save functions ready")

 Save functions ready


In [9]:
# FAILED MODELS CONFIGURATION 

# Only the SqueezeNet models that failed
FAILED_MODELS_CONFIG = [
    {
        'name': 'MM_SqueezeNet_BiLSTM',
        'vision': 'squeezenet1_1',
        'text_encoder': 'bilstm',
        'text_hidden': 64,
        'desc': 'SqueezeNet + Bi-LSTM (small)'
    },
    {
        'name': 'MM_SqueezeNet_GRU',
        'vision': 'squeezenet1_1',
        'text_encoder': 'gru',
        'text_hidden': 64,
        'desc': 'SqueezeNet + GRU (small)'
    },
]

print(" FAILED MODELS TO RETRAIN:")
for i, cfg in enumerate(FAILED_MODELS_CONFIG, 1):
    print(f"   {i}. {cfg['name']:35} | {cfg['desc']}")
print(f"Total: {len(FAILED_MODELS_CONFIG)} models")
print("\n These models will be retrained with SqueezeNet")

 FAILED MODELS TO RETRAIN:
   1. MM_SqueezeNet_BiLSTM                | SqueezeNet + Bi-LSTM (small)
   2. MM_SqueezeNet_GRU                   | SqueezeNet + GRU (small)
Total: 2 models

 These models will be retrained with SqueezeNet


In [10]:
#  MODEL COMPONENTS 

class BiLSTMTextEncoder(nn.Module):
    """Bi-LSTM Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, 2, bidirectional=True, 
                           batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context

class GRUTextEncoder(nn.Module):
    """GRU Text Encoder with Attention."""
    def __init__(self, vocab_size, embed_dim=300, hidden=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden, 2, bidirectional=True, 
                         batch_first=True, dropout=dropout)
        self.attention = nn.Linear(hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.output_dim = hidden * 2
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        gru_out, _ = self.gru(embedded)
        attn_weights = torch.softmax(self.attention(gru_out), dim=1)
        context = torch.sum(attn_weights * gru_out, dim=1)
        return context

def create_squeezenet_encoder(pretrained=True):
    """Create a SqueezeNet encoder that outputs fixed-size features."""
    
    # Load pretrained SqueezeNet
    model = models.squeezenet1_1(pretrained=pretrained)
    
    # Remove the classifier (final conv layers)
    # SqueezeNet's classifier is: (1x1 conv, ReLU, AdaptiveAvgPool2d)
    # We'll replace it with our own pooling
    model.classifier = nn.Identity()
    
    # Create a wrapper with proper pooling
    class SqueezeNetEncoder(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.features = base_model.features  # The feature extractor
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.feature_dim = 512
            
        def forward(self, x):
            # Pass through features
            x = self.features(x)  # Shape: [batch, 512, 13, 13]
            # Global average pooling
            x = self.pool(x)      # Shape: [batch, 512, 1, 1]
            # Flatten
            x = x.view(x.size(0), -1)  # Shape: [batch, 512]
            return x
    
    return SqueezeNetEncoder(model), 512

def get_vision_encoder(name, pretrained=True):
    """Get vision backbone with feature dimension."""
    
    # Special handling for SqueezeNet
    if name == 'squeezenet1_1':
        model, dim = create_squeezenet_encoder(pretrained)
        return model, dim
    
    # Other models
    backbones = {
        'mobilenet_v2': (models.mobilenet_v2, 1280),
        'efficientnet_b0': (models.efficientnet_b0, 1280),
        'shufflenet_v2_x1_0': (models.shufflenet_v2_x1_0, 1024),
    }
    
    if name not in backbones:
        available = list(backbones.keys()) + ['squeezenet1_1']
        raise ValueError(f"Unknown model: {name}. Available: {available}")
    
    model_fn, dim = backbones[name]
    model = model_fn(pretrained=pretrained)
    
    # Remove classification head
    if hasattr(model, 'classifier'):
        model.classifier = nn.Identity()
    elif hasattr(model, 'fc'):
        model.fc = nn.Identity()
    
    # Freeze early layers
    for param in list(model.parameters())[:10]:
        param.requires_grad = False
    
    return model, dim

class MultimodalModel(nn.Module):
    """Multimodal emotion classifier."""
    def __init__(self, vision_name, text_encoder, dropout=0.5):
        super().__init__()
        
        # Vision encoder
        self.vision, vdim = get_vision_encoder(vision_name)
        
        # Vision projection
        self.vision_proj = nn.Sequential(
            nn.Linear(vdim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Text encoder
        self.text_encoder = text_encoder
        
        # Text projection
        self.text_proj = nn.Sequential(
            nn.Linear(text_encoder.output_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion + Classifier
        fusion_dim = 512 + 256
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )
        
    def forward(self, images, texts):
        # Vision features
        v = self.vision(images)      # Should be [batch, 512] after pooling
        v = self.vision_proj(v)       # [batch, 512]
        
        # Text features
        t = self.text_encoder(texts)  # [batch, 256]
        t = self.text_proj(t)         # [batch, 256]
        
        # Fusion
        fused = torch.cat([v, t], dim=1)  # [batch, 768]
        return self.classifier(fused)
    
    def get_parameters(self):
        return self.parameters()

def create_text_encoder(config, vocab_size):
    """Create text encoder based on config."""
    if config['text_encoder'] == 'bilstm':
        return BiLSTMTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    elif config['text_encoder'] == 'gru':
        return GRUTextEncoder(vocab_size, hidden=config.get('text_hidden', 128))
    else:
        raise ValueError(f"Unknown text encoder: {config['text_encoder']}")

print(" Model components ready (FIXED for SqueezeNet!)")

# Test SqueezeNet feature extraction
print("\n Testing SqueezeNet feature extraction...")
try:
    model, dim = create_squeezenet_encoder(pretrained=True)
    print(f" SqueezeNet feature extractor created! Output dim: {dim}")
    
    # Test with dummy input
    dummy = torch.randn(2, 3, 224, 224)
    with torch.no_grad():
        output = model(dummy)
    print(f" Test forward pass successful! Output shape: {output.shape}")
    print(f"   Expected: [2, 512]")
    
except Exception as e:
    print(f" Error: {e}")
    print("   Will use random weights for SqueezeNet")

 Model components ready (FIXED for SqueezeNet!)

 Testing SqueezeNet feature extraction...
 SqueezeNet feature extractor created! Output dim: 512
 Test forward pass successful! Output shape: torch.Size([2, 512])
   Expected: [2, 512]


In [11]:
#  TRAIN FAILED MODELS ONLY (SIMPLIFIED)


print(" RETRAINING SQUEEZENET MODELS ONLY")
print(f"   Models to train: {len(FAILED_MODELS_CONFIG)}")
print(f"   Epochs: {EPOCHS}")
print(f"   Patience: {EARLY_STOPPING_PATIENCE}")

results = []

for config in FAILED_MODELS_CONFIG:
    model_name = config['name']
    
    print(f"# {model_name}")
    print(f"# {config['desc']}")
    
    # Check if already saved
    if model_exists(model_name):
        results.append({
            'model_name': model_name,
            'status': 'skipped',
            'best_accuracy': None
        })
        continue
    
    try:
        # Create text encoder
        text_enc = create_text_encoder(config, vocab_size)
        
        print(f" Creating model with SqueezeNet...")
        model = MultimodalModel(config['vision'], text_enc)
        
        total_params = sum(p.numel() for p in model.parameters()) / 1e6
        print(f" Parameters: {total_params:.2f}M")
        
        # Verify vision output shape
        print(" Verifying vision encoder...")
        dummy_img = torch.randn(2, 3, 224, 224)
        with torch.no_grad():
            v_out = model.vision(dummy_img)
            print(f"   Vision output shape: {v_out.shape}")
            print(f"   Expected: [2, 512]")
        
        # Train
        trained_model, history, best_acc = train_model(
            model=model,
            train_loader=train_loader_custom,
            val_loader=val_loader_custom,
            model_name=model_name,
            epochs=EPOCHS,
            patience=EARLY_STOPPING_PATIENCE
        )
        
        # Save immediately
        save_model(
            model=trained_model,
            optimizer=optim.Adam(model.get_parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY),
            history=history,
            model_name=model_name,
            best_acc=best_acc
        )
        
        results.append({
            'model_name': model_name,
            'best_accuracy': best_acc,
            'status': 'completed'
        })
        
        # Clear memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f" Error: {e}")
        import traceback
        traceback.print_exc()
        results.append({
            'model_name': model_name,
            'status': 'failed',
            'error': str(e),
            'best_accuracy': None
        })


print(" SAVING SUMMARY")
df = pd.DataFrame(results)
df.to_csv(os.path.join(MODEL_DIR, 'squeezenet_training_summary.csv'), index=False)
print(f"\ Summary saved: {MODEL_DIR}squeezenet_training_summary.csv")

 RETRAINING SQUEEZENET MODELS ONLY
   Models to train: 2
   Epochs: 15
   Patience: 3
# MM_SqueezeNet_BiLSTM
# SqueezeNet + Bi-LSTM (small)
 Creating model with SqueezeNet...
 Parameters: 2.56M
 Verifying vision encoder...
   Vision output shape: torch.Size([2, 512])
   Expected: [2, 512]
 TRAINING: MM_SqueezeNet_BiLSTM
   Device: cpu
   Epochs: 15
   Patience: 3


Epoch 1/15 | Train Loss: 0.3283 | Val Loss: 0.3632 | Val Acc: 0.8657
 New best: 0.8657


Epoch 2/15 | Train Loss: 0.1982 | Val Loss: 0.1388 | Val Acc: 0.9473
 New best: 0.9473


Epoch 3/15 | Train Loss: 0.1649 | Val Loss: 0.1369 | Val Acc: 0.9509
 New best: 0.9509


Epoch 4/15 | Train Loss: 0.1375 | Val Loss: 0.1182 | Val Acc: 0.9502
  No improvement (1/3)


...

Epoch 5/15 | Train Loss: 0.0904 | Val Loss: 0.1178 | Val Acc: 0.9560
 New best: 0.9560


Epoch 6/15 | Train Loss: 0.0885 | Val Loss: 0.1234 | Val Acc: 0.9588
 New best: 0.9588


Epoch 7/15 | Train Loss: 0.0800 | Val Loss: 0.1368 | Val Acc: 0.9574
  No improvement (1/3)


Epoch 8/15 | Train Loss: 0.0668 | Val Loss: 0.1236 | Val Acc: 0.9567
  No improvement (2/3)


Epoch 9/15 | Train Loss: 0.0610 | Val Loss: 0.1410 | Val Acc: 0.9581
  No improvement (3/3)
 Early stopping triggered!

 MM_SqueezeNet_GRU COMPLETE!
   Best Accuracy: 0.9588 at epoch 6
   Total epochs: 9
 Model saved: Emotion_Models/MM_SqueezeNet_GRU_final.pt
   Best Accuracy: 0.9588
   File Size: 9.55 MB
 SAVING SUMMARY
\ Summary saved: Emotion_Models/squeezenet_training_summary.csv
